# Chest X-ray Deep Learning: Classification and Detection

Vision Transformer classification and Faster R-CNN detection experiments. Original saved outputs are retained where available. This notebook has not been rerun after documentation cleanup.


This notebook documents the classification and detection experiments. Data and model checkpoints are not included.


## Chest X-ray classification with Vision Transformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DATA_ROOT = '/kaggle/input/chest-xray-pneumonia/chest_xray'
if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    import kagglehub
    DATA_ROOT = os.path.join(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'), 'chest_xray')
assert os.path.isdir(os.path.join(DATA_ROOT, 'train')), DATA_ROOT
print(DATA_ROOT)


In [ ]:
import torch
import torch.nn.functional as F

def Your_attention(Q, K, V, num_head):
    B, N, D = Q.shape
    assert D % num_head == 0
    d = D // num_head
    Q = Q.reshape(B, N, num_head, d).transpose(1, 2)
    K = K.reshape(B, N, num_head, d).transpose(1, 2)
    V = V.reshape(B, N, num_head, d).transpose(1, 2)
    weights = F.softmax((Q @ K.transpose(-2, -1)) / (d ** 0.5), dim=-1)
    return (weights @ V).transpose(1, 2).contiguous().reshape(B, N, D)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super(PatchEmbedding, self).__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.image_size = image_size
        # [TODO] Defining the number of num_patches according to image size and patch size
        self.num_patches = (image_size // patch_size) ** 2
        # [TODO] Using Conv2d for patchification, fill in the parameters
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, D = x.size()
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        out = self.out(Your_attention(Q, K, V, self.num_heads))

        return out

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, feedforward_dim, dropout=0.1):
        super(TransformerEncoderLayer, self).__init__()
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out = self.attn(x)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim, num_heads, num_layers, num_classes, feedforward_dim=2048, dropout=0.1):
        super(VisionTransformer, self).__init__()
        self.patch_embedding = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)

        # [TODO] defining the CLS token in the following.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.patch_embedding.num_patches + 1, embed_dim))
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.pos_embedding, std=0.02)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(embed_dim, num_heads, feedforward_dim, dropout)
            for _ in range(num_layers)
        ])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):

        x = self.patch_embedding(x)

        # [TODO] building your forward process in the following.
        # Hint: You need to append the learnable cls_token before patch embeddings, then pass through encoder layers, and extract the cls_token output for classification with mlp_head.

        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1) + self.pos_embedding
        for layer in self.encoder_layers:
            x = layer(x)
        return self.mlp_head(x[:, 0])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer(224, 16, 3, 384, 8, 8, 2, 1024)
model = model.to(device)

input_tensor = torch.randn(4, 3, 224, 224).to(device)
output = model(input_tensor)
print(output.shape)

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ChestXRayDataset(Dataset):
  def __init__(self, root_dir, transform=None):
      self.root_dir = root_dir
      self.transform = transform
      self.image_paths = []
      self.labels = []

      for label, name in enumerate(['NORMAL', 'PNEUMONIA']):
          folder = os.path.join(root_dir, name)
          for filename in sorted(os.listdir(folder)):
              if filename.lower().endswith(('.jpeg', '.jpg', '.png')):
                  self.image_paths.append(os.path.join(folder, filename))
                  self.labels.append(label)

  def __len__(self):
      return len(self.image_paths)

  def __getitem__(self, idx):

      image = Image.open(self.image_paths[idx]).convert('RGB')
      label = self.labels[idx]


      if self.transform:
          image = self.transform(image)

      return image, label

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

train_dir = os.path.join(DATA_ROOT, 'train')
test_dir = os.path.join(DATA_ROOT, 'test')

train_dataset = ChestXRayDataset(root_dir=train_dir, transform=data_transforms)
test_dataset = ChestXRayDataset(root_dir=test_dir, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

print("Train NORMAL samples:", len([label for label in train_dataset.labels if label == 0]))
print("Train PNEUMONIA samples:", len([label for label in train_dataset.labels if label == 1]))
print("Test NORMAL samples:", len([label for label in test_dataset.labels if label == 0]))
print("Test PNEUMONIA samples:", len([label for label in test_dataset.labels if label == 1]))



In [ ]:
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(torch.tensor([1, 0.4], device=device)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)
# Hint: if you succeed, you will see the printed lr is decreasing

lr_list = []

def train_model(model, criterion, optimizer, num_epochs=5):
  for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader)

    for inputs, labels in progress_bar:
      inputs = inputs.to(device)
      labels = labels.to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      progress_bar.set_postfix(loss=loss.item())

    lr_list.append(optimizer.param_groups[0]['lr'])
    scheduler.step()
    os.makedirs('/content/drive/MyDrive/BN5212_checkpoints', exist_ok=True)
    torch.save(model.state_dict(), '/content/drive/MyDrive/BN5212_checkpoints/vit_latest.pth')

  print(lr_list)
  return model

model = train_model(model, criterion, optimizer, num_epochs=15)
torch.save(model.state_dict(), 'chest_xray_vit.pth')

In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, cohen_kappa_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

def test_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            probs = torch.softmax(outputs, dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    # [TODO] calculate accuracy, precision, recall, kappa.
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    kappa = cohen_kappa_score(all_labels, all_preds)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Cohen Kappa: {kappa:.4f}")

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=["NORMAL", "PNEUMONIA"], yticklabels=["NORMAL", "PNEUMONIA"])
    plt.title("Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0, 1], [0, 1], '--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.show()

model.load_state_dict(torch.load('chest_xray_vit.pth', map_location=device, weights_only=True))
test_model(model, test_loader)



### Controlled optimization experiment


2. **Experiment design:**

**Proposed intervention: class-balanced sampling.** Compare the baseline training procedure against a model trained with `WeightedRandomSampler`, assigning each training example a sampling weight inversely proportional to its class frequency. Keep the ViT architecture, initialization seed, loss function, optimizer, learning rate, schedule, number of epochs, input transforms, and validation/test data unchanged. This changes one factor: training-example sampling. Hypothesis: exposing the model to more NORMAL examples may improve normal-class specificity, although pneumonia recall could decrease.


3. **Experiment:**

Train the baseline and class-balanced-sampling model under matched settings. Record training loss, validation confusion matrices, precision, recall, specificity, Cohen’s kappa, and ROC AUC for both. The code cell below implements the controlled comparison; its output must be generated by running it.


**Execution note:** The following pre-existing code compares weight-decay values and does **not** implement the class-balanced sampling intervention described above. Replace it with a controlled sampling experiment before treating it as evidence for the written hypothesis.


In [ ]:
import copy
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score

indices = np.arange(len(train_dataset))
tr_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=train_dataset.labels)
research_train = DataLoader(Subset(train_dataset, tr_idx), batch_size=16, shuffle=True, num_workers=2)
research_val = DataLoader(Subset(train_dataset, val_idx), batch_size=16, shuffle=False, num_workers=2)

@torch.no_grad()
def research_metrics(net, loader):
    net.eval()
    true, predicted = [], []
    for images, labels in loader:
        logits = net(images.to(device))
        predicted.extend(logits.argmax(1).cpu().tolist())
        true.extend(labels.tolist())
    return {'accuracy': accuracy_score(true, predicted), 'recall': recall_score(true, predicted, zero_division=0)}

research_results = []
for weight_decay in [0.0, 0.01]:
    torch.manual_seed(42)
    net = VisionTransformer(224, 16, 3, 384, 8, 8, 2, 1024).to(device)
    opt = optim.AdamW(net.parameters(), lr=0.0001, weight_decay=weight_decay)
    for epoch in range(3):
        net.train()
        for images, labels in tqdm(research_train, desc=f'weight_decay={weight_decay} epoch={epoch+1}'):
            opt.zero_grad(set_to_none=True)
            loss = criterion(net(images.to(device)), labels.to(device))
            loss.backward()
            opt.step()
    result = {'weight_decay': weight_decay, **research_metrics(net, research_val)}
    research_results.append(result)
    print(result)
print('Research comparison:', research_results)


4. **Analysis and Interpretation:**

**Complete after execution.** Baseline specificity: **[value]**; sampled-model specificity: **[value]**. Baseline pneumonia recall: **[value]**; sampled-model recall: **[value]**. Baseline AUC: **[value]**; sampled-model AUC: **[value]**. The change in false positives was **[value]** and the change in false negatives was **[value]**. Conclude whether balanced sampling addressed the observed baseline weakness and discuss the clinical trade-off between missed pneumonia and unnecessary follow-up. Do not claim improvement unless the recorded results support it.


## Chest X-ray abnormality detection with Faster R-CNN


**Step 0: Required setup**


Environment preparation

In [ ]:
!pip install torch torchvision tqdm matplotlib pillow scipy

In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
import os, json

from torch.utils.data import Dataset, DataLoader
from PIL import Image


# In this dataset, we only detect consolidation
CLASSES = ["Consolidation"]
CLS2ID = {c: i+1 for i, c in enumerate(CLASSES)}

class ChestXDet10(Dataset):
    def __init__(self, base, split="train", transforms=None):
        self.base = base
        self.split = split
        self.transforms = transforms
        self.img_dir = f"{base}/{split}"
        self.ann = self.read_json(f"{base}/{split}.json")

        self.items = []
        for it in self.ann:
            p = self._try_read(os.path.join(self.img_dir, it["file_name"]))
            if p: self.items.append((p, it))
        print(f"[{split}] usable images:", len(self.items))


    def read_json(self, path):
        with open(path, "r") as f:
            return json.load(f)

    def _try_read(self, img_path):
        if os.path.exists(img_path):
            return img_path
        alt = img_path.replace(".png",".PNG")
        return alt if os.path.exists(alt) else None

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):

        '''
            Expected output:
            img: tensor of shape (3, H, W)
            target: dict of the following keys:
                boxes: tensor of shape (N, 4)  # [x1, y1, x2, y2]
                labels: tensor of shape (N)  # [1, 2, ..., 10]
            img_path: the path of the image
        '''

        img_path, it = self.items[idx]
        # Please notice that in our dataset, there exist image without any boxes.
        # The label for consolidation is 1 and background is 0.

        # [TODO]: use Image from PIL package to load image
        #  - the loaded image should be named "img"
        img = Image.open(img_path).convert('RGB')

        w, h = img.size # get the width and height of the image

        syms  = it.get("syms", [])
        boxes = np.array(it.get("boxes", []), dtype=np.float32).reshape(-1, 4) if syms else np.zeros((0,4), dtype=np.float32)

        # clamp
        if len(boxes):
            boxes[:, 0::2] = np.clip(boxes[:, 0::2], 0, w - 1)
            boxes[:, 1::2] = np.clip(boxes[:, 1::2], 0, h - 1)

        #
        labels = np.array([CLS2ID[s] for s in syms], dtype=np.int64) if syms else np.zeros((0,), dtype=np.int64)

        # [TODO]: convert boxes and labels to torch tensors.
        #  - You need to choose suitable data type for boxes and labels
        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target, os.path.basename(img_path)

def collate_fn(batch):
    imgs, targets, names = list(zip(*batch))
    return list(imgs), list(targets), list(names)


# The base directory of your data
BASE = '/content/drive/MyDrive/Assignment_task2'

train_tfms = T.Compose([T.ToTensor()])
test_tfms  = T.Compose([T.ToTensor()])

train_set = ChestXDet10(BASE, "train", transforms=train_tfms)
test_set  = ChestXDet10(BASE, "test",  transforms=test_tfms)

# Here we use batch size 4 due to the memory limit
train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_set,  batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead, RegionProposalNetwork
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.generalized_rcnn import GeneralizedRCNN
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection._utils import overwrite_eps
from torchvision.models.detection.backbone_utils import (
    _resnet_fpn_extractor,
    _validate_trainable_layers,
)
from torchvision.models.resnet import resnet50
from torchvision.ops import misc as misc_nn_ops
from torch.hub import load_state_dict_from_url


class FasterRCNN(GeneralizedRCNN):
    """
    Implements Faster R-CNN.

    The input to the model is expected to be a list of tensors, each of shape [C, H, W], one for each
    image, and should be in 0-1 range. Different images can have different sizes.

    The behavior of the model changes depending if it is in training or evaluation mode.

    During training, the model expects both the input tensors, as well as a targets (list of dictionary),
    containing:
        - boxes (``FloatTensor[N, 4]``): the ground-truth boxes in ``[x1, y1, x2, y2]`` format, with
          ``0 <= x1 < x2 <= W`` and ``0 <= y1 < y2 <= H``.
        - labels (Int64Tensor[N]): the class label for each ground-truth box

    The model returns a Dict[Tensor] during training, containing the classification and regression
    losses for both the RPN and the R-CNN.

    During inference, the model requires only the input tensors, and returns the post-processed
    predictions as a List[Dict[Tensor]], one for each input image. The fields of the Dict are as
    follows:
        - boxes (``FloatTensor[N, 4]``): the predicted boxes in ``[x1, y1, x2, y2]`` format, with
          ``0 <= x1 < x2 <= W`` and ``0 <= y1 < y2 <= H``.
        - labels (Int64Tensor[N]): the predicted labels for each image
        - scores (Tensor[N]): the scores or each prediction

    Args:
        backbone (nn.Module): the network used to compute the features for the model.
            It should contain a out_channels attribute, which indicates the number of output
            channels that each feature map has (and it should be the same for all feature maps).
            The backbone should return a single Tensor or and OrderedDict[Tensor].
        num_classes (int): number of output classes of the model (including the background).
            If box_predictor is specified, num_classes should be None.
        min_size (int): minimum size of the image to be rescaled before feeding it to the backbone
        max_size (int): maximum size of the image to be rescaled before feeding it to the backbone
        image_mean (Tuple[float, float, float]): mean values used for input normalization.
            They are generally the mean values of the dataset on which the backbone has been trained
            on
        image_std (Tuple[float, float, float]): std values used for input normalization.
            They are generally the std values of the dataset on which the backbone has been trained on
        rpn_anchor_generator (AnchorGenerator): module that generates the anchors for a set of feature
            maps.
        rpn_head (nn.Module): module that computes the objectness and regression deltas from the RPN
        rpn_pre_nms_top_n_train (int): number of proposals to keep before applying NMS during training
        rpn_pre_nms_top_n_test (int): number of proposals to keep before applying NMS during testing
        rpn_post_nms_top_n_train (int): number of proposals to keep after applying NMS during training
        rpn_post_nms_top_n_test (int): number of proposals to keep after applying NMS during testing
        rpn_nms_thresh (float): NMS threshold used for postprocessing the RPN proposals
        rpn_fg_iou_thresh (float): minimum IoU between the anchor and the GT box so that they can be
            considered as positive during training of the RPN.
        rpn_bg_iou_thresh (float): maximum IoU between the anchor and the GT box so that they can be
            considered as negative during training of the RPN.
        rpn_batch_size_per_image (int): number of anchors that are sampled during training of the RPN
            for computing the loss
        rpn_positive_fraction (float): proportion of positive anchors in a mini-batch during training
            of the RPN
        rpn_score_thresh (float): during inference, only return proposals with a classification score
            greater than rpn_score_thresh
        box_roi_pool (MultiScaleRoIAlign): the module which crops and resizes the feature maps in
            the locations indicated by the bounding boxes
        box_head (nn.Module): module that takes the cropped feature maps as input
        box_predictor (nn.Module): module that takes the output of box_head and returns the
            classification logits and box regression deltas.
        box_score_thresh (float): during inference, only return proposals with a classification score
            greater than box_score_thresh
        box_nms_thresh (float): NMS threshold for the prediction head. Used during inference
        box_detections_per_img (int): maximum number of detections per image, for all classes.
        box_fg_iou_thresh (float): minimum IoU between the proposals and the GT box so that they can be
            considered as positive during training of the classification head
        box_bg_iou_thresh (float): maximum IoU between the proposals and the GT box so that they can be
            considered as negative during training of the classification head
        box_batch_size_per_image (int): number of proposals that are sampled during training of the
            classification head
        box_positive_fraction (float): proportion of positive proposals in a mini-batch during training
            of the classification head
        bbox_reg_weights (Tuple[float, float, float, float]): weights for the encoding/decoding of the
            bounding boxes
    """

    def __init__(
        self,
        backbone,
        num_classes=None,
        # transform parameters
        min_size=800,
        max_size=1333,
        image_mean=None,
        image_std=None,
        # RPN parameters
        rpn_anchor_generator=None,
        rpn_head=None,
        rpn_pre_nms_top_n_train=2000,
        rpn_pre_nms_top_n_test=1000,
        rpn_post_nms_top_n_train=2000,
        rpn_post_nms_top_n_test=1000,
        rpn_nms_thresh=0.7,
        rpn_fg_iou_thresh=0.7,
        rpn_bg_iou_thresh=0.3,
        rpn_batch_size_per_image=256,
        rpn_positive_fraction=0.5,
        rpn_score_thresh=0.0,
        # Box parameters
        box_roi_pool=None,
        box_head=None,
        box_predictor=None,
        box_score_thresh=0.05,
        box_nms_thresh=0.5,
        box_detections_per_img=100,
        box_fg_iou_thresh=0.5,
        box_bg_iou_thresh=0.5,
        box_batch_size_per_image=512,
        box_positive_fraction=0.25,
        bbox_reg_weights=None,
    ):
        if not hasattr(backbone, "out_channels"):
            raise ValueError(
                "backbone should contain an attribute out_channels "
                "specifying the number of output channels (assumed to be the "
                "same for all the levels)"
            )

        assert isinstance(rpn_anchor_generator, (AnchorGenerator, type(None)))
        assert isinstance(box_roi_pool, (MultiScaleRoIAlign, type(None)))

        out_channels = backbone.out_channels

        if rpn_anchor_generator is None:
            anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
            aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
            rpn_anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)
        if rpn_head is None:
            rpn_head = RPNHead(out_channels, rpn_anchor_generator.num_anchors_per_location()[0])

        rpn_pre_nms_top_n = dict(training=rpn_pre_nms_top_n_train, testing=rpn_pre_nms_top_n_test)
        rpn_post_nms_top_n = dict(training=rpn_post_nms_top_n_train, testing=rpn_post_nms_top_n_test)

        rpn = RegionProposalNetwork(
            rpn_anchor_generator,
            rpn_head,
            rpn_fg_iou_thresh,
            rpn_bg_iou_thresh,
            rpn_batch_size_per_image,
            rpn_positive_fraction,
            rpn_pre_nms_top_n,
            rpn_post_nms_top_n,
            rpn_nms_thresh,
            score_thresh=rpn_score_thresh,
        )

        if box_roi_pool is None:
            box_roi_pool = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=7, sampling_ratio=2)

        if box_head is None:
            resolution = box_roi_pool.output_size[0]
            representation_size = 1024
            box_head = TwoMLPHead(out_channels * resolution ** 2, representation_size)

        if box_predictor is None:
            representation_size = 1024
            box_predictor = FastRCNNPredictor(representation_size, num_classes)

        roi_heads = RoIHeads(
            # Box
            box_roi_pool,
            box_head,
            box_predictor,
            box_fg_iou_thresh,
            box_bg_iou_thresh,
            box_batch_size_per_image,
            box_positive_fraction,
            bbox_reg_weights,
            box_score_thresh,
            box_nms_thresh,
            box_detections_per_img,
        )

        if image_mean is None:
            image_mean = [0.485, 0.456, 0.406]
        if image_std is None:
            image_std = [0.229, 0.224, 0.225]
        transform = GeneralizedRCNNTransform(min_size, max_size, image_mean, image_std)

        super().__init__(backbone, rpn, roi_heads, transform)


class TwoMLPHead(nn.Module):
    """
    Standard heads for FPN-based models

    Args:
        in_channels (int): number of input channels
        representation_size (int): size of the intermediate representation
    """



    def __init__(self, in_channels, representation_size):
        super().__init__()

        # [TODO]: Define the layers of the head
        # Hints:
        # - This head is a simple 2-layer MLP (fully-connected -> ReLU -> fully-connected -> ReLU).
        # - Create TWO nn.Linear layers:
        #     1) fc6 maps from `in_channels` to `representation_size`.
        #     2) fc7 maps from `representation_size` to `representation_size` (keeps the same size).
        # - Name the layers exactly `self.fc6` and `self.fc7` to match common checkpoints and code that may load them.
        self.fc6 = nn.Linear(in_channels, representation_size)
        self.fc7 = nn.Linear(representation_size, representation_size)


    def forward(self, x):

        # [TODO]: Implement the forward pass
        # Hints:
        # - Use the layers you created before, implement the forward as a simple 2-layer MLP (fully-connected -> ReLU -> fully-connected -> ReLU).


        return F.relu(self.fc7(F.relu(self.fc6(x.flatten(start_dim=1)))))


class FastRCNNPredictor(nn.Module):
    """
    Standard classification + bounding box regression layers
    for Fast R-CNN.

    Args:
        in_channels (int): number of input channels
        num_classes (int): number of output classes (including background)
    """

    def __init__(self, in_channels, num_classes):
        super().__init__()


        # [TODO]: Define the layers of the predictor
        # Hints:
        # - Create two fully-connected (nn.Linear) layers sharing the SAME input features
        #     * Classification head (cls_score): outputs class logits of shape (N, num_classes).
        #       return raw scores (logits).
        #     * Each class (including background) gets 4 box deltas [dx, dy, dw, dh].
        self.cls_score = nn.Linear(in_channels, num_classes)
        self.bbox_pred = nn.Linear(in_channels, num_classes * 4)
        #       Each class (including background) gets 4 box deltas [dx, dy, dw, dh].




    def forward(self, x):

        # [TODO]: Implement the forward pass
        # Hints:
        # - use input x and your created layers to get scores (bbox classification score) and bbox_deltas
        if x.dim() == 4:
            assert list(x.shape[2:]) == [1, 1]

        x = x.flatten(start_dim=1)

        scores = self.cls_score(x)
        bbox_deltas = self.bbox_pred(x)

        return scores, bbox_deltas


model_urls = {
    "fasterrcnn_resnet50_fpn_coco": "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth",
    "fasterrcnn_mobilenet_v3_large_320_fpn_coco": "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth",
    "fasterrcnn_mobilenet_v3_large_fpn_coco": "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth",
}



def fasterrcnn_resnet50_fpn(
    pretrained=False, progress=True, num_classes=91, pretrained_backbone=True, trainable_backbone_layers=None, **kwargs
):
    """
    Constructs a Faster R-CNN model with a ResNet-50-FPN backbone.

    Reference: `"Faster R-CNN: Towards Real-Time Object Detection with
    Region Proposal Networks" <https://arxiv.org/abs/1506.01497>`_.

    The input to the model is expected to be a list of tensors, each of shape ``[C, H, W]``, one for each
    image, and should be in ``0-1`` range. Different images can have different sizes.

    The behavior of the model changes depending if it is in training or evaluation mode.

    During training, the model expects both the input tensors, as well as a targets (list of dictionary),
    containing:

        - boxes (``FloatTensor[N, 4]``): the ground-truth boxes in ``[x1, y1, x2, y2]`` format, with
          ``0 <= x1 < x2 <= W`` and ``0 <= y1 < y2 <= H``.
        - labels (``Int64Tensor[N]``): the class label for each ground-truth box

    The model returns a ``Dict[Tensor]`` during training, containing the classification and regression
    losses for both the RPN and the R-CNN.

    During inference, the model requires only the input tensors, and returns the post-processed
    predictions as a ``List[Dict[Tensor]]``, one for each input image. The fields of the ``Dict`` are as
    follows, where ``N`` is the number of detections:

        - boxes (``FloatTensor[N, 4]``): the predicted boxes in ``[x1, y1, x2, y2]`` format, with
          ``0 <= x1 < x2 <= W`` and ``0 <= y1 < y2 <= H``.
        - labels (``Int64Tensor[N]``): the predicted labels for each detection
        - scores (``Tensor[N]``): the scores of each detection

    For more details on the output, you may refer to :ref:`instance_seg_output`.

    Faster R-CNN is exportable to ONNX for a fixed batch size with inputs images of fixed size.

    Args:
        pretrained (bool): If True, returns a model pre-trained on COCO train2017
        progress (bool): If True, displays a progress bar of the download to stderr
        num_classes (int): number of output classes of the model (including the background)
        pretrained_backbone (bool): If True, returns a model with backbone pre-trained on Imagenet
        trainable_backbone_layers (int): number of trainable (not frozen) resnet layers starting from final block.
            Valid values are between 0 and 5, with 5 meaning all backbone layers are trainable. If ``None`` is
            passed (the default) this value is set to 3.
    """
    trainable_backbone_layers = _validate_trainable_layers(
        pretrained or pretrained_backbone, trainable_backbone_layers, 5, 3
    )

    if pretrained:
        # no need to download the backbone if pretrained is set
        pretrained_backbone = False

    backbone = resnet50(pretrained=pretrained_backbone, progress=progress, norm_layer=misc_nn_ops.FrozenBatchNorm2d)
    backbone = _resnet_fpn_extractor(backbone, trainable_backbone_layers)
    model = FasterRCNN(backbone, num_classes, **kwargs)
    if pretrained:
        state_dict = load_state_dict_from_url(model_urls["fasterrcnn_resnet50_fpn_coco"], progress=progress)
        model.load_state_dict(state_dict)
        overwrite_eps(model, 0.0)
    return model




device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = fasterrcnn_resnet50_fpn(pretrained_backbone=True, num_classes=len(CLASSES) + 1)


In [ ]:
from tqdm import tqdm
import torch
import torch.nn as nn

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=0.00005, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    pbar = tqdm(enumerate(data_loader), total=len(data_loader))

    total_loss = 0.0

    for i, (images, targets, _) in pbar:

        # [TODO] implement the training loop

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        loss_value = losses.item()

        # todo2: zero grad the optimizer
        optimizer.zero_grad(set_to_none=True)

        # todo3: backward to compute the gradient
        losses.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # todo4: optimize the param in the model
        optimizer.step()

        total_loss += loss_value



        pbar.set_postfix(loss=loss_value)
    avg_loss = total_loss / len(data_loader)
    return avg_loss


EPOCHS = 5  # Here we only train for 5 epochs
for epoch in range(EPOCHS):
    avg_loss = train_one_epoch(model, optimizer, train_loader, device)
    print(f"Epoch {epoch} loss: {avg_loss}")
    lr_scheduler.step()
    torch.save(model.state_dict(), f"{BASE}/fasterrcnn_latest.pth")

# save the model
WEIGHTS = f"{BASE}/fasterrcnn_chestxdet10.pth"
torch.save(model.state_dict(), WEIGHTS)
print("saved to:", WEIGHTS)

In [ ]:
import torch
import json, os
from tqdm import tqdm

model.eval()
pred_dir = f"{BASE}/pred_json"
os.makedirs(pred_dir, exist_ok=True)

score_thresh = 0.02

with torch.no_grad():
    for images, targets, names in tqdm(test_loader):
        # [TODO] implement the model inference and generate the prediction
        images_gpu = [img.to(device) for img in images]
        outputs = model(images_gpu)

        for name, out, im in zip(names, outputs, images):
            boxes = out["boxes"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            labels = out["labels"].cpu().numpy()

            per_image = []
            for (x1,y1,x2,y2), sc, lb in zip(boxes, scores, labels):
                if sc < score_thresh:
                    continue
                # label==0 background；TorchVision's label>=1 corresponds to our 0..9 classes
                if lb == 0:
                    continue
                cls_name = CLASSES[lb-1]
                per_image.append({cls_name: [float(x1), float(y1), float(x2), float(y2), float(sc)]})

            out_path = os.path.join(pred_dir, os.path.splitext(name)[0] + ".json")
            with open(out_path, "w") as f:
                json.dump(per_image, f)




len(os.listdir(pred_dir))

In [ ]:
import numpy as np
import json
import os

from scipy import interpolate



def IOU(box1, gts):
    # compute overlaps
    # intersection

    # compute the intersection over union

    # The bounding box coordinates are defined as closed intervals,
    # i.e., both the top-left (x1, y1) and bottom-right (x2, y2) points are inclusive, so the width is (x2 - x1 + 1) and the height is (y2 - y1 + 1).
    ixmin = np.maximum(gts[:, 0], box1[0])
    iymin = np.maximum(gts[:, 1], box1[1])
    ixmax = np.minimum(gts[:, 2], box1[2])
    iymax = np.minimum(gts[:, 3], box1[3])

    # [TODO] compute the intersection width (iw) and intersection height (ih)
    iw = np.maximum(ixmax - ixmin + 1.0, 0.0)
    ih = np.maximum(iymax - iymin + 1.0, 0.0)

    inters = iw * ih

    # union
    uni = ((box1[2] - box1[0] + 1.) * (box1[3] - box1[1] + 1.) +
           (gts[:, 2] - gts[:, 0] + 1.) *
           (gts[:, 3] - gts[:, 1] + 1.) - inters)

    overlaps = inters / uni


    return overlaps

def FROC(boxes_all, gts_all, iou_th):
    # Compute the FROC curve, for single class only
    nImg = len(boxes_all)
    img_idxs_ori = np.hstack([[i]*len(boxes_all[i]) for i in range(nImg)]).astype(int)
    boxes_cat = np.vstack(boxes_all)
    scores = boxes_cat[:, -1]
    ord = np.argsort(scores)[::-1]
    boxes_cat = boxes_cat[ord, :4]
    img_idxs = img_idxs_ori[ord]

    hits = [np.zeros((len(gts),), dtype=bool) for gts in gts_all]
    nHits = 0
    nMiss = 0
    tps = []
    fps = []
    precisions = []

    for i in range(len(boxes_cat)):
        overlaps = IOU(boxes_cat[i, :], gts_all[img_idxs[i]])
        if overlaps.shape[0] == 0:
            nMiss += 1
        elif overlaps.max() < iou_th:
            nMiss += 1
        else:
            max_index = np.argmax(overlaps)
            if overlaps[max_index] >= iou_th and not hits[img_idxs[i]][max_index]:
                hits[img_idxs[i]][max_index] = True
                nHits += 1
            elif overlaps[max_index] >= iou_th and hits[img_idxs[i]][max_index]:
                nMiss += 1

        tps.append(nHits)
        fps.append(nMiss)
        if nHits + nMiss > 0:
            precisions.append(nHits / (nHits + nMiss))
        else:
            precisions.append(0.0)

    nGt = len(np.vstack(gts_all))
    sens = np.array(tps, dtype=float) / nGt
    fp_per_img = np.array(fps, dtype=float) / nImg
    precisions = np.array(precisions, dtype=float)

    if len(tps) != 0:
        tp_gt = [tps[-1], nGt]
    else:
        tp_gt = None

    return tp_gt, sens, fp_per_img, precisions


def eval_FROC(gt_boxes, all_boxes, CLASSES, avgFP=[0.05, 0.1, 0.2, 0.5], iou_th=0.5):
    mean_recall = [0.0 for _ in range(len(avgFP) + 1)]
    mean_precision = [0.0 for _ in range(len(avgFP) + 1)]

    for cls in range(len(all_boxes)):
        tp_gt, sens, fp_per_img, precisions = FROC(all_boxes[cls], gt_boxes[cls], iou_th)

        # interpolate precision and recall
        if len(fp_per_img) > 1:
            f_recall = interpolate.interp1d(fp_per_img, sens, fill_value="extrapolate")
            f_precision = interpolate.interp1d(fp_per_img, precisions, fill_value="extrapolate")
        else:
            continue

        for idx, fp in enumerate(avgFP):
            if fp <= fp_per_img[-1]:
                mean_recall[idx] += float(f_recall(fp))
                mean_precision[idx] += float(f_precision(fp))

    print('='*20, 'Recall & Precision', '='*20)
    for i, fp in enumerate(avgFP):
        print('Recall@%.2f=%.2f%%, Precision@%.2f=%.2f%%' %
              (fp, mean_recall[i]/len(CLASSES)*100.0,
               fp, mean_precision[i]/len(CLASSES)*100.0))


def sens_at_FP(boxes_all, gts_all, avgFP, iou_th):
    # compute the sensitivity at avgFP (average FP per image)
    tp_gt, sens, fp_per_img, _ = FROC(boxes_all, gts_all, iou_th)

    # gt = tp and no fp
    if len(fp_per_img)!= 0 and sens[-1] == 1 and fp_per_img[-1] == 0:
        res = np.ones(1)
        valid_avgFP = np.zeros(1)

        return res, valid_avgFP

    if len(fp_per_img) != 0:
        max_fp = fp_per_img[-1]

        # tp = 0 and fp = 1
        if len(fp_per_img) == 1:
            fp_per_img = np.insert(fp_per_img, 0, 0.0)
            sens = np.insert(sens, 0, 0.0)

        if max_fp != 0:
            f = interpolate.interp1d(fp_per_img, sens, fill_value='extrapolate')
        if(avgFP[-1] < max_fp):
            valid_avgFP_end_idx = len(avgFP)
        else:
            valid_avgFP_end_idx = np.argwhere(np.array(avgFP) >= max_fp)[0][0]
        valid_avgFP = np.hstack((avgFP[:valid_avgFP_end_idx], max_fp))
        if max_fp != 0:
            res = f(valid_avgFP)
        # no fp and tp < gt
        elif max_fp == 0 and tp_gt:
            res = np.zeros(len(valid_avgFP))
            res[:] = 1.0*tp_gt[0] / tp_gt[1]
    # no tp
    else:
        res = np.zeros(1)
        valid_avgFP = np.zeros(1)

    return res, valid_avgFP


def get_pred_info(pred_dir, file_id_list, CLASSES):
    class_num = len(CLASSES)
    all_boxes = [[] for _ in range(class_num)]

    for cls_idx, class_name in enumerate(CLASSES):
        for file_id in file_id_list[cls_idx]:
            file_id = file_id.replace('.png', '.json')
            json_file = os.path.join(pred_dir, file_id)
            all_box = []
            if os.path.exists(json_file) and os.stat(json_file).st_size > 0:
                annos = json.load(open(json_file))
                for i in range(len(annos)):
                    if class_name in annos[i].keys():
                        all_box.append(annos[i][class_name])
                    elif class_name.lower() in annos[i].keys():
                        all_box.append(annos[i][class_name.lower()])
            if all_box:
                all_box = np.array(all_box, dtype=np.float32)
            else:
                all_box = np.zeros((0, 5), dtype=np.float32)
            all_boxes[cls_idx].append(all_box)

    return all_boxes

def get_gt_info(test_json, CLASSES):
    class_num = len(CLASSES)
    gt_boxes = [[] for _ in range(class_num)]
    file_id_list = [[] for _ in range(class_num)]

    annos = json.load(open(test_json))
    for cls_idx, class_name in enumerate(CLASSES):
        for entry in annos:
            syms, boxes = entry['syms'], entry['boxes']
            file_name = entry['file_name']
            gt_box = []
            for idx, (sym, box) in enumerate(zip(syms, boxes)):
                if sym == class_name:
                    gt_box.append(box)
            if gt_box:
                gt_box = np.array(gt_box, dtype=np.float32)
            else:
                gt_box = np.zeros((0, 4), dtype=np.float32)
            gt_boxes[cls_idx].append(gt_box)
            file_id_list[cls_idx].append(file_name)

    return gt_boxes, file_id_list


test_json = BASE + "/test.json"
pred_dir = BASE + "/pred_json"

gt_boxes, file_id_list = get_gt_info(test_json, CLASSES)
all_boxes = get_pred_info(pred_dir, file_id_list, CLASSES)
eval_FROC(gt_boxes, all_boxes, CLASSES)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import ImageDraw

def draw_boxes_pil(img, boxes, labels, classes, scores=None,
                   color="red", text_color="white"):
    # function to draw the boxes on the image

    img = img.copy()
    draw = ImageDraw.Draw(img)

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

        if len(labels) > 0:
            lb = int(labels[i])
            if 1 <= lb <= len(classes):
                name = classes[lb - 1]
            else:
                name = f"cls{lb}"
        else:
            name = ""

        if scores is not None:
            txt = f"{name}:{float(scores[i]):.2f}"
        else:
            txt = name if name else "GT"

        tw = len(txt) * 6
        th = 12
        draw.rectangle([x1, max(0, y1 - th), x1 + tw + 4, y1], fill=color)
        draw.text((x1+2, y1 - th), txt, fill=text_color)

    return img

@torch.no_grad()
def viz_pred_gt_fixed(model, dataset, classes, filenames, score_thresh=0.3, device=None):
    """
    This method will visualize the predictions and ground truth for the given filenames in the dataset.
    You should plot 4 images in a 2x4 grid. The top row will be the predictions and the bottom row will be the ground truth.
    Args:
        model: the model to be visualized
        dataset: the custom dataset class
        classes: the classes to be visualized
        filenames: the filenames to be visualized
        score_thresh: the threshold for the confidence score
        device: the device to be used
    Returns:
        None
    """

    model.eval()
    device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

    cols = len(filenames)
    fig, axes = plt.subplots(2, cols, figsize=(4*cols, 8))
    if cols == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    # Find the index of the given filenames in the dataset
    name2idx = {os.path.basename(it["file_name"]): i for i, (_,it) in enumerate(dataset.items)} \
               if hasattr(dataset, "items") else {dataset[i][2]: i for i in range(len(dataset))}

    for ci, fname in enumerate(filenames):
        idx = name2idx[fname]
        img_tensor, target, basename = dataset[idx]
        img_pil = T.ToPILImage()(img_tensor)

        # Model inference
        inp = img_tensor.to(device)
        out = model([inp])[0]

        boxes_p = out["boxes"].cpu().numpy()
        scores_p = out["scores"].cpu().numpy()
        labels_p = out["labels"].cpu().numpy()
        keep = scores_p >= score_thresh
        boxes_p, scores_p, labels_p = boxes_p[keep], scores_p[keep], labels_p[keep]

        # GT
        boxes_g = target["boxes"].numpy()
        labels_g = target["labels"].numpy()

        # [TODO] Use draw_boxes_pil function to draw the predictions and ground truth
        # requirement:
        #   pred_vis should be red
        #   gt_vis should be green
        pred_vis = draw_boxes_pil(img_pil, boxes_p, labels_p, classes, scores=scores_p, color="red")
        gt_vis = draw_boxes_pil(img_pil, boxes_g, labels_g, classes, color="green")

        axes[0, ci].imshow(pred_vis); axes[0, ci].set_title(f"Pred: {basename}")
        axes[0, ci].axis("off")
        axes[1, ci].imshow(gt_vis);   axes[1, ci].set_title("Ground Truth")
        axes[1, ci].axis("off")

    plt.tight_layout()
    plt.show()




# Visualize the predictions and ground truth for the first 4 images in the test set
filenames = ["40424.png", "41225.png", "62137.png", "45701.png"]

viz_pred_gt_fixed(model, test_set, CLASSES, filenames, score_thresh=0.3, device=device)



### Controlled optimization experiment


2. **Experiment design:**

**Proposed intervention: anchor design.** Derive candidate anchor sizes and aspect ratios from the training bounding boxes only, then replace the baseline RPN anchor configuration while keeping the backbone, image preprocessing, loss, optimizer, learning rate, schedule, epoch count, and evaluation protocol fixed. Evaluate both models on the same held-out images at the same score and IoU thresholds. Hypothesis: anchors better matched to consolidation boxes may increase lesion recall, potentially at the expense of more false positives.


3. **Experiment:**

Train and evaluate baseline and modified-anchor Faster R-CNN models using the same split and training budget. Record lesion recall at IoU ≥ 0.5, average false positives per image, and example predictions overlaid with ground-truth boxes. **The existing code cell below currently compares learning rates, not anchors, and must be replaced before this is a valid controlled experiment.**


In [ ]:
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

indices = np.arange(len(train_set))
research_tr, research_val = train_test_split(indices, test_size=0.2, random_state=42)
research_train_loader = DataLoader(Subset(train_set, research_tr), batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)
research_val_loader = DataLoader(Subset(train_set, research_val), batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)

@torch.no_grad()
def detection_recall_at_iou(net, loader, threshold=0.5):
    net.eval()
    hits = total = 0
    for images, targets, _ in tqdm(loader):
        outputs = net([img.to(device) for img in images])
        for pred, target in zip(outputs, targets):
            gt = target['boxes']
            total += len(gt)
            keep = pred['scores'].cpu() >= 0.3
            predicted = pred['boxes'].cpu()[keep]
            if len(gt) and len(predicted):
                overlaps = torchvision.ops.box_iou(gt, predicted)
                hits += int((overlaps.max(dim=1).values >= threshold).sum().item())
    return hits / total if total else 0.0

research_results_detection = []
for lr in [0.00005, 0.0001]:
    torch.manual_seed(42)
    net = fasterrcnn_resnet50_fpn(pretrained_backbone=True, num_classes=len(CLASSES) + 1).to(device)
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    for epoch in range(2):
        loss = train_one_epoch(net, opt, research_train_loader, device)
        print('LR', lr, 'epoch', epoch + 1, 'loss', loss)
    recall = detection_recall_at_iou(net, research_val_loader)
    research_results_detection.append({'lr': lr, 'validation_recall_iou_0.5': recall})
    print(research_results_detection[-1])
print('Detection research comparison:', research_results_detection)


4. **Analysis and Interpretation:**

**Complete after execution.** Baseline lesion recall at IoU ≥ 0.5: **[value]**; modified-anchor recall: **[value]**. Baseline false positives/image: **[value]**; modified-anchor false positives/image: **[value]**. Compare matched ground-truth and predicted boxes and explain whether the modified anchors improved detection of previously missed consolidations. Discuss the effect on false positives and whether the intervention addressed the original diagnosis. Do not invent experimental outcomes.
